# NGLab Tutorial #3: Trading Environment

Learn how the Rust `TradingEnv` provides a Gymnasium-compatible interface for reinforcement learning.

## Learning Objectives

1. Create a `TradingEnv` instance
2. Understand the step lifecycle
3. Execute actions and receive rewards
4. Visualize agent performance

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque

# Try importing Rust environment
try:
    from nglab import TradingEnv, EnvConfig
    RUST_AVAILABLE = True
    print("✓ Rust TradingEnv imported")
except ImportError:
    RUST_AVAILABLE = False
    print("⚠ Using Python simulation")

# Standard RL libraries
try:
    import gymnasium as gym
    print("✓ Gymnasium available")
except ImportError:
    print("⚠ Gymnasium not installed (pip install gymnasium)")

## 1. The Step Lifecycle

Every RL environment follows the same pattern:

In [ ]:
obs, info = env.reset()
done = False

while not done:
    action = agent.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

### NGLab's Step Breakdown

1. **Action**: Agent sends action (0: Hold, 1: Buy, 2: Sell)
2. **Execution**: OrderBook calculates slippage
3. **State Update**: Portfolio updates cash/position
4. **Observation**: Returns numpy array of recent prices
5. **Reward**: $ R_t = \text{Returns}_t - \text{Costs} - \text{DrawdownPenalty} $

## 2. Creating the Environment

In [ ]:
if RUST_AVAILABLE:
    # Create configuration
    config = EnvConfig(
        initial_cash=100000.0,
        lookback_window=60,
        transaction_cost=0.001,  # 0.1% per trade
        slippage_base=0.0005,    # 5bps
    )
    
    env = TradingEnv(config)
    obs, info = env.reset(seed=42)
    
    print(f"Environment created successfully!")
    print(f"Observation shape: {obs.shape}")
    print(f"Action space: {env.action_space}")
    print(f"Initial cash: ${env.cash:,.2f}")
else:
    # Simulated environment
    print("Creating simulated environment...")
    
    class MockEnv:
        def __init__(self):
            self.cash = 100000.0
            self.position = 0.0
            self.step_count = 0
            self.prices = 50000 + np.cumsum(np.random.randn(1000) * 100)
        
        def reset(self):
            self.step_count = 0
            self.cash = 100000.0
            self.position = 0.0
            return self._get_obs(), {}
        
        def _get_obs(self):
            start = max(0, self.step_count - 60)
            return self.prices[start:self.step_count+1][-60:]
        
        def step(self, action):
            self.step_count += 1
            price = self.prices[self.step_count]
            
            # Simple logic: 0=hold, 1=buy 0.1 BTC, 2=sell 0.1 BTC
            if action == 1 and self.cash >= price * 0.1:
                self.position += 0.1
                self.cash -= price * 0.1
            elif action == 2 and self.position >= 0.1:
                self.position -= 0.1
                self.cash += price * 0.1
            
            portfolio_value = self.cash + self.position * price
            reward = (portfolio_value - 100000) / 100000  # Normalized return
            
            done = self.step_count >= len(self.prices) - 1
            return self._get_obs(), reward, done, False, {'portfolio_value': portfolio_value}
    
    env = MockEnv()
    obs, info = env.reset()
    print(f"Mock environment created")
    print(f"Observation shape: {obs.shape}")

## 3. Running a Random Agent

In [ ]:
# Run simulation for 1000 steps
n_steps = 1000
rewards_history = []
portfolio_values = []
actions_taken = []

obs, info = env.reset()

for step in range(n_steps):
    # Random agent (uniform distribution)
    action = np.random.choice([0, 1, 2])  # Hold, Buy, Sell
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    rewards_history.append(reward)
    portfolio_values.append(info.get('portfolio_value', env.cash if not RUST_AVAILABLE else 0))
    actions_taken.append(action)
    
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

print(f"\n=== Simulation Results ===")
print(f"Total steps: {len(rewards_history)}")
print(f"Cumulative reward: {sum(rewards_history):.4f}")
print(f"Final portfolio value: ${portfolio_values[-1]:,.2f}")
print(f"Return: {((portfolio_values[-1] / portfolio_values[0]) - 1) * 100:.2f}%")

## 4. Analyzing Agent Behavior

In [ ]:
# Create visualizations
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Portfolio value over time
axes[0].plot(portfolio_values, linewidth=2, color='steelblue')
axes[0].axhline(y=portfolio_values[0], color='red', linestyle='--', label='Initial Value')
axes[0].set_title('Portfolio Value Over Time', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Value (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative rewards
cumulative_rewards = np.cumsum(rewards_history)
axes[1].plot(cumulative_rewards, linewidth=2, color='green')
axes[1].set_title('Cumulative Rewards', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cumulative Reward')
axes[1].grid(True, alpha=0.3)

# Action distribution
action_names = ['Hold', 'Buy', 'Sell']
action_counts = [actions_taken.count(i) for i in range(3)]
axes[2].bar(action_names, action_counts, color=['gray', 'green', 'red'], alpha=0.7)
axes[2].set_title('Action Distribution', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Zero-Copy Observations

NGLab uses **zero-copy memory sharing** between Rust and Python:

In [ ]:
// Rust allocates
let obs_buffer: Vec<f64> = vec![0.0; lookback * features];

// Create Python array pointing to Rust memory
let py_array = PyArray1::from_vec(py, obs_buffer);

This avoids copying ~30KB of data on every step, reducing overhead by ~30%.

In [ ]:
# Demonstrate observation structure
obs, _ = env.reset()

print(f"Observation shape: {obs.shape}")
print(f"Data type: {obs.dtype}")
print(f"Memory size: {obs.nbytes / 1024:.2f} KB")
print(f"\nFirst 5 values: {obs[:5]}")
print(f"Last 5 values: {obs[-5:]}")

## 6. Reward Function Breakdown

The reward is calculated as:

$$
R_t = \underbrace{\frac{PV_t - PV_{t-1}}{PV_{t-1}}}_{\text{Returns}} - \underbrace{\mathbb{1}_{\text{trade}} \cdot c}_{\text{Costs}} - \underbrace{\max(0, \text{DD}_t - \theta)}_{\text{Penalty}}
$$

Where:
- $PV_t$ = Portfolio value at time $t$
- $c$ = Transaction cost (e.g., 0.1%)
- $DD_t$ = Current drawdown
- $\theta$ = Drawdown threshold

In [ ]:
# Analyze reward components
rewards_array = np.array(rewards_history)

print("\n=== Reward Statistics ===")
print(f"Mean reward: {rewards_array.mean():.6f}")
print(f"Std reward: {rewards_array.std():.6f}")
print(f"Max reward: {rewards_array.max():.6f}")
print(f"Min reward: {rewards_array.min():.6f}")
print(f"Sharpe ratio: {rewards_array.mean() / (rewards_array.std() + 1e-8):.4f}")

## Summary

In this notebook, you learned:

✅ The Gymnasium-compatible step lifecycle  
✅ Creating and configuring a `TradingEnv`  
✅ Running a random agent and analyzing performance  
✅ Zero-copy memory optimization  
✅ Reward function components  

## Next Steps

Continue to **Notebook #4**: Time-Series Forecasting to add predictive models!

---